# Exploration des couches transports (IDFM)

Fichiers produits par `scripts/03_download_transports.py`.

In [1]:
import geopandas as gpd, pandas as pd
cur = gpd.read_file('../data/raw/transports/emplacement-des-gares-idf.geojson')
prj = gpd.read_file('../data/raw/transports/projets_arrets_idf.geojson')
cur.shape, prj.shape, cur.crs.to_string()

((1240, 28), (301, 23), 'EPSG:4326')

## 1. Réseau actuel : une ligne par couple gare × ligne

In [2]:
cur.drop(columns=['geometry', 'geo_point_2d']).head(3).T

,0,1,2
objectid_1,91,94,128
id_gares,196,207,307
nom_gares,Combs-la-Ville - Quincy,Corbeil-Essonnes,Gare de Lyon
nom_so_gar,NaN,NaN,NaN
nom_su_gar,NaN,NaN,NaN
id_ref_zdc,62558,60309,73626
nom_zdc,Combs-la-Ville - Quincy,Corbeil-Essonnes,Gare de Lyon
id_ref_zda,45771,43115,470195
nom_zda,Combs-la-Ville - Quincy,Corbeil-Essonnes,Gare de Lyon
idrefliga,A01842,A01842,A01842


In [3]:
cur.groupby('mode').res_com.agg(['size', 'nunique', lambda s: sorted(s.unique())])

,size,nunique,<lambda_0>
mode,,,
CABLE,5,1,[CABLE 1]
METRO,405,16,"[METRO 1, METRO 10, METRO 11, METRO 12, METRO ..."
RER,252,5,"[RER A, RER B, RER C, RER D, RER E]"
TRAIN,285,10,"[GL, TRAIN H, TRAIN J, TRAIN K, TRAIN L, TRAIN..."
TRAM,5,1,[TRAM 14]
TRAMWAY,278,14,"[TRAM 1, TRAM 10, TRAM 11, TRAM 12, TRAM 13, T..."
VAL,10,3,"[CDGVAL, FUNICULAIRE MONTMARTRE, ORLYVAL]"


In [4]:
print('doublons (gare, ligne) :', cur.duplicated(['nom_gares', 'res_com']).sum())
print('gares distinctes :', cur.id_ref_zdc.nunique(), '| hors IDF (idf=0) :', (cur.idf == 0).sum())
sorted(cur[cur.res_com == 'METRO 14'].nom_gares)

doublons (gare, ligne) : 0
gares distinctes : 996 | hors IDF (idf=0) : 52


["Aéroport d'Orly",
 'Bercy',
 'Bibliothèque François Mitterrand',
 'Chevilly - Larue',
 'Châtelet',
 'Cour Saint-Émilion',
 'Gare de Lyon',
 'Hôpital Bicêtre',
 "L'Haÿ - les - Roses",
 'Madeleine',
 'Mairie de Saint-Ouen',
 'Maison Blanche',
 'Olympiades',
 'Pont Cardinet',
 'Porte de Clichy',
 'Pyramides',
 'Saint-Denis – Pleyel',
 'Saint-Lazare',
 'Saint-Ouen',
 'Thiais - Orly',
 'Villejuif - Gustave Roussy']

## 2. Projets : Grand Paris Express et autres

In [5]:
prj.groupby(['mode', 'nom_projet', 'statut']).size().to_frame('arrets')

arrets
mode  nom_projet         statut                   
bus   TZen2              Travaux                29
      TZen3              Travaux                21
      TZen5              Travaux                19
      bus Altival        Arrêté de DUP           7
                         Travaux                 4
      bus Bords de Marne Arrêté de DUP          19
      bus Senia - Orly   AVP                     9
      bus entre Seine    AVP                    34
métro metro 15 est       Travaux                13
      metro 15 ouest     Travaux                12
      metro 15 sud       Travaux                16
      metro 16           Travaux                10
      metro 17           Travaux                10
      metro 18           Travaux                11
train RER E - Ouest      Travaux                12
tram  Tram 01 - Colombes Travaux                12
      Tram 01 - Rueil    AVP                    16
      Tram 01 - VDF      Travaux                16
      Tram 04 - CM       Etudes préalables       1
                         Travaux                 1
      Tram 07            Travaux                 7
      Tram 08            AVP                    11
      Tram 11            Etudes préalables       7
      Tram 13            Travaux                 4

In [6]:
gpe = prj[prj['mode'] == 'métro']
print(len(gpe), 'lignes ;', gpe.nom_arret.nunique(), 'gares distinctes')
print(gpe.groupby('res_com').size().to_dict())
gpe.groupby('nom_arret').res_com.apply(lambda s: sorted(set(s))).loc[lambda s: s.str.len() > 1]

72 lignes ; 60 gares distinctes
{'METRO 15': 41, 'METRO 16': 10, 'METRO 17': 10, 'METRO 18': 11}


nom_arret
La Courneuve-Six Routes              [METRO 16, METRO 17]
Le Bourget                           [METRO 16, METRO 17]
Noisy-Champs                         [METRO 15, METRO 16]
Saint-Denis Pleyel         [METRO 15, METRO 16, METRO 17]
Name: res_com, dtype: object

In [7]:
prj.columns.tolist()  # aucune colonne de date de mise en service ; dup_date = date de déclaration d'utilité publique

['nom_arret',
 'id_projet',
 'nom_projet',
 'id_operati',
 'operation',
 'nature',
 'mode',
 'sous_mode',
 'indice',
 'cor_exist',
 'terminus',
 'phase',
 'variante',
 'creation',
 'prolong',
 'amelior',
 'statut',
 'dup',
 'dup_date',
 'opendata',
 'res_com',
 'geo_point_2d',
 'geometry']